<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

# import library

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
#Convert generate drum track data into MIDI format
# Output file are placed in folder "output_midi"

import librosa, IPython, datetime, time, os, sys, copy, glob, pickle
import numpy as np
from time import gmtime, strftime
from IPython.display import Image
import pypianoroll
import matplotlib.pyplot as plt
%matplotlib inline
from pathlib import Path
import pypianoroll as ppr   # alias used below

# show version info
print ("[info] Current Time:     " + datetime.datetime.now().strftime('%Y/%m/%d  %H:%M:%S'))
print ("[info] Python Version:   " + sys.version.split('\n')[0].split(' ')[0])
print ("[info] Working Dir:      " + os.getcwd()+'/')


# === NEW: Config that aligns with Wei et al. (2019) pipeline ===
# Wei: SSMs are 256×256 (zero-padded to 256 bars) and drums are 16th-note quantized (46×16 per bar).
# We will (1) read your step_4 predictions (B,46,16), (2) upsample to pypianoroll’s 96 steps/bar,
# (3) map 46 drum classes back to GM pitches, and (4) write MIDI into ./output_midi.

# I/O roots (update only if you moved things)
ALL_TRACKS_DIR = Path("./pre_processed_data/proc_all_tracks_mid")           # from step_1
PRED_DIRS = [
    Path("./pre_processed_data/drum_generation/predictions"),               # step_4 typical
    Path("./pre_processed_data/generation/drum_predictions"),               # alt
    Path("./output_data/drum_predictions"),                                  # alt
]
OUT_DIR = Path("./output_midi")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# If step_1 saved a mapping, load it; otherwise fallback to a standard 46-GM mapping
# (Note: this MUST match the mapping used during training; if your step_1 saved a mapping file,
# we auto-load it; if not, this default works for a typical 46-class GM set.)
def try_load_drum_mapping():
    candidates = [
        Path("./pre_processed_data/metadata/drum_index_to_midi.json"),
        Path("./pre_processed_data/metadata/drum_index_to_pitch.json"),
        Path("./pre_processed_data/metadata/drum_pitch_map.json"),
        Path("./metadata/drum_index_to_midi.json"),
    ]
    import json
    for fp in candidates:
        if fp.exists():
            with open(fp, "r") as f:
                arr = json.load(f)
            # allow {"0":36, ...} or [36,...]
            if isinstance(arr, dict):
                # ensure order by integer key
                items = sorted(((int(k), v) for k,v in arr.items()), key=lambda x: x[0])
                return [int(v) for _,v in items]
            return [int(x) for x in arr]
    # Fallback 46-class GM pitch set (drop 35, keep 36..81 subset commonly used)
    return list(range(36, 82)) # ensure length==46

DRUM_PITCHES = try_load_drum_mapping()
assert len(DRUM_PITCHES) == 46, f"Expected 46 drum classes, got {len(DRUM_PITCHES)}"

# Wei uses 96 steps per bar for spectrogram time (CQT) and 16 steps per bar for symbolic drums.
STEPS_PER_BAR_DRUM = 16
STEPS_PER_BAR_PPR  = 96  # pypianoroll beat_resolution=24 -> 4 beats * 24 = 96
UPSAMPLE = STEPS_PER_BAR_PPR // STEPS_PER_BAR_DRUM  # = 6

# Try to find your step_4 prediction files
def discover_prediction_files():
    preds = []
    for root in PRED_DIRS:
        if root.exists():
            preds += list(root.glob("*.npz"))
            preds += list(root.glob("*.pkl"))
    # Back-compat with Wei’s original name
    preds += list(Path(".").glob("model_result_binary_list.pkl"))
    # Deduplicate
    uniq = []
    seen = set()
    for p in preds:
        if p.resolve() not in seen:
            uniq.append(p)
            seen.add(p.resolve())
    return uniq

def load_prediction_any(fp: Path):
    """
    Returns: dict with keys:
      - 'song_id': str
      - 'drum_bin': np.ndarray of shape (B, 46, 16) binary {0,1}
    Tries several layouts used by step_4 / older Wei code.
    """
    import pickle
    if fp.suffix == ".npz":
        with np.load(fp, allow_pickle=True) as d:
            # common keys
            for k in ("drum_bin", "drum_binary", "pred", "y_hat", "drum"):
                if k in d:
                    arr = d[k]
                    break
            else:
                # try a single array
                arr = list(d.values())[0]
    elif fp.suffix == ".pkl":
        with open(fp, "rb") as f:
            obj = pickle.load(f)
        # Could be a dict or a plain array or Wei’s list keyed by pitch version
        if isinstance(obj, dict):
            # try common keys
            for k in ("drum_bin", "drum_binary", "pred", "y_hat", "drum"):
                if k in obj:
                    arr = obj[k]
                    break
            else:
                # Wei’s: model_result_binary_list[pch_ver][bar_idx,:,:]
                # try first value
                arr = next(iter(obj.values()))
        else:
            arr = np.asarray(obj)
    else:
        raise ValueError(f"Unsupported prediction file: {fp}")

    arr = np.asarray(arr)
    # Normalize shape to (B,46,16)
    if arr.ndim == 3:
        B,C,T = arr.shape
        if (C,T) == (46,16):
            drum_bin = arr
        elif (B,T,C) == (46,16,arr.shape[2]):  # transpose guess
            drum_bin = np.transpose(arr, (2,0,1))
        elif (C,B) == (46,16):
            drum_bin = np.transpose(arr, (1,0,2))
        elif (T,C) == (16,46):
            drum_bin = np.transpose(arr, (0,2,1))
        else:
            raise ValueError(f"Unexpected pred shape {arr.shape} in {fp}")
    else:
        raise ValueError(f"Expected 3D array, got {arr.shape} in {fp}")

    # song id from filename
    song_id = fp.stem
    return {"song_id": song_id, "drum_bin": drum_bin.astype(np.uint8)}

def upsample_16_to_96(bar_46x16: np.ndarray) -> np.ndarray:
    """(46,16)->(46,96) by repeating each 16th step 6 times."""
    assert bar_46x16.shape == (46,16)
    return np.repeat(bar_46x16, UPSAMPLE, axis=1)

def build_drum_track(drum_B_46x16: np.ndarray) -> ppr.Track:
    """
    drum_B_46x16: (B,46,16) binary
    Returns a pypianoroll Track with shape (B*96, 128) and GM drum pitches set.
    """
    B = drum_B_46x16.shape[0]
    pr = np.zeros((B * STEPS_PER_BAR_PPR, 128), dtype=np.uint8)
    for b in range(B):
        up = upsample_16_to_96(drum_B_46x16[b])  # (46,96)
        t0 = b * STEPS_PER_BAR_PPR
        # Place each class at its GM pitch
        for k, pitch in enumerate(DRUM_PITCHES):
            # simple on/off; velocity 100 for 'on' frames
            on = up[k] > 0
            pr[t0:t0+STEPS_PER_BAR_PPR, pitch] = np.where(on, 100, pr[t0:t0+STEPS_PER_BAR_PPR, pitch])
    return ppr.Track(pianoroll=pr, program=0, is_drum=True, name="Drums_Generated")

def load_all_tracks(song_id: str) -> ppr.Multitrack | None:
    # Your step_1 wrote: ./pre_processed_data/proc_all_tracks_mid/{file}_all_tracks.mid
    # Try a few common patterns.
    cands = list(ALL_TRACKS_DIR.glob(f"{song_id}*all_tracks.mid")) + \
            list(ALL_TRACKS_DIR.glob(f"{song_id}.mid")) + \
            list(ALL_TRACKS_DIR.glob(f"*{song_id}*.mid"))
    for p in cands:
        try:
            return ppr.Multitrack(str(p))
        except Exception:
            continue
    return None

def write_outputs(song_id: str, drum_track: ppr.Track, mt_all: ppr.Multitrack | None):
    # (A) drum-only
    drum_only = ppr.Multitrack(tracks=[drum_track], tempo=np.array([[120.0]]), beat_resolution=24)
    p_drums = OUT_DIR / f"{song_id}__drum_only.mid"
    drum_only.write(str(p_drums))
    print(f"[info] saved {p_drums.name}")

    # (B) merge with all-tracks (replace any existing drum track)
    if mt_all is not None:
        # normalize length
        T = drum_track.pianoroll.shape[0]
        for tr in mt_all.tracks:
            if tr.pianoroll is not None:
                if tr.pianoroll.shape[0] < T:
                    pad = np.zeros((T - tr.pianoroll.shape[0], tr.pianoroll.shape[1]), dtype=tr.pianoroll.dtype)
                    tr.pianoroll = np.vstack([tr.pianoroll, pad])
                else:
                    tr.pianoroll = tr.pianoroll[:T]

        # drop old drums
        kept = [tr for tr in mt_all.tracks if not getattr(tr, "is_drum", False)]
        merged = ppr.Multitrack(tracks=kept + [drum_track], tempo=mt_all.tempo, beat_resolution=mt_all.beat_resolution)
        p_merged = OUT_DIR / f"{song_id}__all_tracks_with_gen_drums.mid"
        merged.write(str(p_merged))
        print(f"[info] saved {p_merged.name}")
    else:
        print(f"[warn] no all-tracks MIDI found for {song_id}; skipped merged output.")

def clamp_or_pad_to_256_bars(drum_B_46x16: np.ndarray) -> np.ndarray:
    """Wei uses 256-bar SSMs (zero-padded). If you want strict parity, enforce 256 bars here."""
    B = drum_B_46x16.shape[0]
    if B == 256:
        return drum_B_46x16
    if B > 256:
        return drum_B_46x16[:256]
    pad = np.zeros((256 - B, 46, 16), dtype=drum_B_46x16.dtype)
    return np.concatenate([drum_B_46x16, pad], axis=0)

# === NEW: main run ===
pred_files = discover_prediction_files()
assert len(pred_files) > 0, "No prediction files found. Make sure step_4_generate_drum.py wrote .npz or .pkl predictions."

for fp in pred_files:
    item = load_prediction_any(fp)
    song_id = item["song_id"]
    drum_bin = clamp_or_pad_to_256_bars(item["drum_bin"])  # shape (256,46,16) if shorter it gets zero-padded
    drum_track = build_drum_track(drum_bin)
    mt_all = load_all_tracks(song_id)
    write_outputs(song_id, drum_track, mt_all)

print(f"[info] All {len(pred_files)} file(s) are saved into {OUT_DIR.resolve()}.")


[info] Current Time:     2025/08/28  15:54:24
[info] Python Version:   3.11.13
[info] Working Dir:      /workspace/


AssertionError: No prediction files found. Make sure step_4_generate_drum.py wrote .npz or .pkl predictions.

# Ensure File DIR function

In [ ]:
def ensure_dir(file_path):
    ed_directory = os.path.dirname(file_path)
    if not os.path.exists(ed_directory):
        os.makedirs(ed_directory)

# Read all song/bar index code

In [ ]:
with open('./pre_processed_data/abs_bar_idx_str_list.pkl', 'rb') as pkl_file:      
    abs_bar_idx_str_list = pickle.load(pkl_file)
    
print ('[info] List of [song/bar] data is loaded.')
print ('[info] Total bars: {}'.format(len(abs_bar_idx_str_list)))
print ('[info] First 5 bar code: {}'.format(abs_bar_idx_str_list[:5]))
print ('[info] Last  5 bar code: {}'.format(abs_bar_idx_str_list[-5:]))


# Define function to get complete single song index (start, end)
song_index_in_list = np.unique([x.split('_')[0] for x in abs_bar_idx_str_list]).tolist()

def get_test_song_abs_idx(pick_song_index):

    song_index_all_bars = [x for x in abs_bar_idx_str_list if x[0:5]==song_index_in_list[pick_song_index]]
    bar_idx_start = abs_bar_idx_str_list.index(song_index_all_bars[0])
    bar_idx_end = abs_bar_idx_str_list.index(song_index_all_bars[-1])
    
    return ([bar_idx_start, bar_idx_end+1])


# for get_song_idx in range(0, 3):
for get_song_idx in range(len(song_index_in_list)):
    print('[info] Song idx: {:2d},   Start:{:4d},   End: {}'.format(get_song_idx,
                                                                    get_test_song_abs_idx(get_song_idx)[0],
                                                                    get_test_song_abs_idx(get_song_idx)[1]))

# Reload all test result

In [ ]:
# heavy

model_result_flist = np.sort(glob.glob('./model_out_result_add_note_*.pkl', recursive=True)).tolist()

model_result_binary_list = []
add_note_ver_list = []

for model_result_file in model_result_flist:

    with open(model_result_file, 'rb') as pkl_file:
        model_result_pkg = pickle.load(pkl_file)
        
    model_result_binary = np.where(model_result_pkg[2] > 0.5,
                                   np.ones_like(model_result_pkg[2]),
                                   np.zeros_like(model_result_pkg[2]))
    
    print ('[info] \'{}\' is reloaded.'.format(model_result_file))
    print ('[info] Data shape: {}'.format(model_result_binary.shape))
        
    model_result_binary_list.append(model_result_binary)
    
    add_note_ver = model_result_file.split('.')[-2][-2:]
    add_note_ver_list.append(add_note_ver)

print ('\n[info] {} files are reloaded.'.format(len(model_result_flist)))


# Reload all original MIDI object

In [ ]:
class midi_track(object):
    def __init__(self):
        self.file_name = ""
        self.pmidi_data = []
        self.pmidi_all_tracks_data = []
        self.pmidi_no_drum_data = []
        self.pmidi_drum_only_data = []
        self.tempo = 0        
        self.downbeats_list_fixed = []
        self.bar_range_list_fixed = []
        self.drum_bar_list = []
        self.drum_bar_list_bin = []
        self.drum_bar_note_num = []
#print ('MIDI track object is defined.')

obj_file_name = './pre_processed_data/proc_midi_object.pkl'
with open(obj_file_name, 'rb') as pkl_file:
    midi_obj_list = pickle.load(pkl_file)
    
print('[info] All MIDI objects: {}'.format(len(midi_obj_list)))

# define original MIDI drum rebuild function (96, 128)

In [ ]:
# keep 99 % of all instrument count (total 46 insts)
selected_inst_list_46 = [27, 28, 33, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, \
                         51, 53, 54, 55, 56, 57, 59, 60, 61, 62, 63, 64, 65, 67, 68, 69, 70, 73, \
                         74, 75, 76, 77, 80, 81, 82, 83, 85, 87]
print ('[info] # of keeped Insts: {}'.format(len(selected_inst_list_46)))

def get_odrum_shape(drum_ary_in):    
    odrum_data = np.zeros([96, 128])
    for x in range(0, drum_ary_in.shape[0]):
        for y in range(0, drum_ary_in.shape[1]):            
            pix_value = drum_ary_in[x,y]
            if pix_value>0.5:
                odrum_data[y*6, selected_inst_list_46[x]] = 100
            
    return (odrum_data)

print ('[info] get_odrum_shape is defined.')

# load original midi data

In [ ]:
all_tracks_mid_flist = np.sort(glob.glob('./input_midi/my_input/*.mid', recursive=True)).tolist()
all_tracks_mid_flist = np.sort([x.replace(' ','') for x in all_tracks_mid_flist if "all_tracks.mid" in x]).tolist()
print ('[info] Total files: {}'.format(len(all_tracks_mid_flist)))
for x in all_tracks_mid_flist[:]: print ('  ' + x)

# Loop all songs and save corresponding MIDI files

In [ ]:
for x_idx, pick_song_index in enumerate(song_index_in_list):

    print ('[info] Start processing song: {} ...'.format(x_idx+1))

    # get complete single song index data
    abs_idx_start, abs_idx_end = get_test_song_abs_idx(x_idx)
    
    abs_song_idx = pick_song_index

    print ('[info] Song index: {}'.format(abs_song_idx))
    print ('[info] Song bars: {}'.format(abs_idx_end - abs_idx_start))
    print ('[info] start\end bar index:  {}\{}'.format(abs_idx_start, abs_idx_end))
    #print ('[info] Abs end index: {}'.format(abs_idx_end))
    #print('')

    # plot complete single song drum arrangement
    bar_idx_start = abs_idx_start
    bar_idx_end = abs_idx_end

    model_darr_odrm_ary_list = []
    
    for pch_ver in range(0, len(model_result_binary_list)):
    
        model_darr_list = []

        for bar_idx in range(bar_idx_start, bar_idx_end):
        
            plot_model_out_darr = model_result_binary_list[pch_ver][bar_idx,:,:]
        
            model_darr_list.append(plot_model_out_darr)
        

        # convert drum data into original shape (96, 128)
        model_darr_odrm_list = [get_odrum_shape(x) for x in model_darr_list]
        model_darr_odrm_ary = np.concatenate(model_darr_odrm_list, axis=0)
        #print(model_darr_odrm_ary.shape)

        model_darr_odrm_ary_list.append(model_darr_odrm_ary)

    
    #Get original NPZ file name
    original_midi_file_path = all_tracks_mid_flist[x_idx]
    # pypiano_obj = pypianoroll.parse(original_midi_file_path, beat_resolution=24, name='original_track')
    pypiano_obj = pypianoroll.parse(original_midi_file_path)
    #ptymidi_obj = pypiano_obj.to_pretty_midi()
    mtrack_data = pypiano_obj
    
    for pch_idx in range(0, len(model_result_binary_list)):
        
        # write drum notes in multitrack object
        mtrack_data.append_track(track=None, 
                                 pianoroll=model_darr_odrm_ary_list[pch_idx], 
                                 program=pch_idx+1, 
                                 is_drum=True,
                                 name='Drums_{}'.format(add_note_ver_list[pch_idx]))

    # transfer data into pretty midi format
    pmidi_data = mtrack_data.to_pretty_midi(constant_tempo=None)

    # print instruments
    print ('[info] Show {} Insts...'.format(len(pmidi_data.instruments)))
    for x in pmidi_data.instruments:
        print ('[info] MIDI ' + str(x))
    print('')

    # make all notes in Drums2 velocity=99
    for instrument in pmidi_data.instruments:
        #if instrument.program==5:
        if instrument.is_drum:
            for note in instrument.notes:
                note.velocity = 120
        else:
            for note in instrument.notes:
                note.velocity = 50            


    song_name_tmp = all_tracks_mid_flist[x_idx].split('/')[-1][:-15] + '_merged'
                
    # set midi file name to write
    midi_file_name = './output_midi/{}.mid'.format(song_name_tmp)

    # create folder if not exist
    ensure_dir(midi_file_name)

    # write midi file
    pmidi_data.write(midi_file_name)
    print ('[info] \"{}\" is saved.\n\n'.format(midi_file_name))
    
print ('[info] All {} files are saved.'.format(len(song_index_in_list)))

# Congratulation ! Now you can find fusion tracks(Original midi + generated drums) under "./output_midi/"

In [ ]:
!ls ./output_midi/

# Use any DAW you like to open the MIDI file, you can see five generated tracks as following.

In [ ]:
Image(url="./track22_bj.png",width=1200,height=800)